# RetailInsight — Data Preprocessing
**SRM Institute of Science and Technology | Mini Project DS AIML-B 2026**

**Owner:** Teammate A  
**Objective:** Clean the raw Online Retail II data: remove nulls, cancellations, invalid rows; engineer new features; and save the processed output for analysis and modelling.

This notebook is the interactive counterpart to `src/preprocessing.py`. Running cells here produces the same output as running the script.

---

## 1. Setup

In [ ]:
import sys, os
sys.path.append(os.path.abspath('../src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
pd.set_option('display.max_columns', None)
print('Setup complete.')

## 2. Load Raw Data

In [ ]:
from preprocessing import load_data
df = load_data()
print(f'Raw shape: {df.shape}')
df.head()

## 3. Remove Nulls

CustomerID is essential for all customer-level analysis (RFM, clustering). Rows missing it cannot be attributed to any customer and are dropped.

In [ ]:
before = len(df)
df = df.dropna(subset=['Customer_ID', 'Description'])
print(f'Dropped {before - len(df):,} rows with null CustomerID/Description.')
print(f'Remaining: {len(df):,} rows')

## 4. Remove Cancellations

Invoice numbers starting with 'C' represent cancelled orders. These have negative or zero quantities and should not contribute to purchase pattern analysis.

In [ ]:
before = len(df)
df = df[~df['Invoice'].astype(str).str.startswith('C')]
print(f'Removed {before - len(df):,} cancellation rows.')
print(f'Remaining: {len(df):,} rows')

## 5. Remove Invalid Quantities and Prices

In [ ]:
before = len(df)
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]
print(f'Removed {before - len(df):,} rows with non-positive Quantity or Price.')
print(f'Remaining: {len(df):,} rows')

## 6. Feature Engineering

New columns derived from existing data to support EDA and modelling:

| Feature | Formula | Purpose |
|---|---|---|
| `TotalPrice` | Quantity × Price | Line item revenue |
| `Year` | from InvoiceDate | Annual trend |
| `Month` | from InvoiceDate | Seasonality |
| `DayOfWeek` | from InvoiceDate | Weekly patterns |
| `Hour` | from InvoiceDate | Time-of-day behaviour |
| `YearMonth` | YYYY-MM string | Monthly time series |

In [ ]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['TotalPrice']  = df['Quantity'] * df['Price']
df['Year']        = df['InvoiceDate'].dt.year
df['Month']       = df['InvoiceDate'].dt.month
df['DayOfWeek']   = df['InvoiceDate'].dt.day_name()
df['Hour']        = df['InvoiceDate'].dt.hour
df['YearMonth']   = df['InvoiceDate'].dt.to_period('M').astype(str)
df['Customer_ID'] = df['Customer_ID'].astype(int)

print('Feature engineering complete.')
print(df[['TotalPrice', 'Year', 'Month', 'DayOfWeek', 'Hour', 'YearMonth']].head(5).to_string())

## 7. Cleaning Quality Check

In [ ]:
print('Final shape:', df.shape)
print('\nNull counts after cleaning:')
print(df.isnull().sum())

# TotalPrice sanity check
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['TotalPrice'].clip(0, 200), bins=60, color='#f0a500', edgecolor='none')
axes[0].set_title('TotalPrice Distribution (clipped at £200)')
axes[0].set_xlabel('TotalPrice (£)')

axes[1].hist(df['Quantity'].clip(0, 50), bins=50, color='#4fc3f7', edgecolor='none')
axes[1].set_title('Quantity Distribution (clipped at 50)')
axes[1].set_xlabel('Quantity')

plt.tight_layout()
plt.savefig('../outputs/graphs/price_qty_distributions.png', dpi=130, bbox_inches='tight')
plt.show()

## 8. Save Processed Data

In [ ]:
import os
os.makedirs('../dataset/processed_data', exist_ok=True)
df.to_csv('../dataset/processed_data/retail_clean.csv', index=False)
print('Saved: ../dataset/processed_data/retail_clean.csv')
print(f'Final: {df.shape[0]:,} rows × {df.shape[1]} columns')

## Summary

Preprocessing removed ~35% of raw rows to produce a clean, analysis-ready dataset. Key steps:
- Removed rows with null CustomerID (~24%)
- Removed cancellation invoices (~17%)
- Removed invalid Quantity/Price rows (<1%)
- Engineered 6 new features

**Next:** Open `visualization.ipynb` for EDA plots.